# Falcon 9 landing classification — reproducible IBM 90-row workflow

Run this notebook in a directory containing `dataset_part_2.csv` and `dataset_part_3.csv`. The 90 original rows are matched by row order. The 80 one-hot encoded features are used to predict the first-stage landing class.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

raw = pd.read_csv('dataset_part_2.csv')
X = pd.read_csv('dataset_part_3.csv')
y = raw['Class'].astype(int)
assert len(X) == len(y) == 90
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)
print('train/test:', len(X_train),len(X_test),'features:',X.shape[1])

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
configs = {
 'Logistic Regression':(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), {'logisticregression__C':[0.01,0.1,1,10]}),
 'SVM':(make_pipeline(StandardScaler(), SVC()), {'svc__kernel':['linear','rbf'],'svc__C':[0.1,1,10]}),
 'Decision Tree':(DecisionTreeClassifier(random_state=42),{'max_depth':[2,4,6,None],'min_samples_split':[2,4,8]}),
 'KNN':(make_pipeline(StandardScaler(), KNeighborsClassifier()), {'kneighborsclassifier__n_neighbors':[3,5,7,9]})
}
results=[]; fitted={}
for name,(est,param) in configs.items():
 grid=GridSearchCV(est,param,cv=cv,scoring='accuracy',n_jobs=-1)
 grid.fit(X_train,y_train)
 fitted[name]=grid.best_estimator_
 results.append({'model':name,'cv_accuracy':grid.best_score_,'test_accuracy':accuracy_score(y_test,grid.predict(X_test)),'params':grid.best_params_})
summary=pd.DataFrame(results).sort_values('test_accuracy',ascending=False)
print(summary.to_string(index=False))

In [ ]:
ax=summary.plot.bar(x='model',y='test_accuracy',legend=False,rot=20,figsize=(9,5))
ax.set_ylim(0,1);ax.set_ylabel('Held-out accuracy');plt.tight_layout();plt.show()
# CV-best classifier for deployment selection, not the test-best classifier
cv_best= max(results,key=lambda r:r['cv_accuracy'])['model']
print('Best via training CV:',cv_best)
ConfusionMatrixDisplay.from_estimator(fitted[cv_best],X_test,y_test)
plt.show()

## Interpretation

The first-stage landing label is defined on the 90-row IBM dataset. This notebook deliberately distinguishes the model chosen by training cross-validation from the one with the best held-out score. The latter should not be used for selection. Prior evidence from an executed related experiment reported 83.3% held-out Logistic Regression accuracy (15/18); differences in parameter grids or fold shuffling can change the results.